# MINGLE on Coherent — Kaggle GPU

Runs the **full** project: vanilla BCE → weighted BCE (50 ep) → focal loss → note ablation.

## Before you press Run All

1. **Accelerator: GPU** (P100/T4). Internet **on** (only for pip). The GitHub repo is **private** — do not git clone.
2. Add **two** Kaggle datasets:
   - `mingle-processed` (tensors)
   - `mingle-code` (the `src/` + `configs/` folder from this project)
3. Add a Kaggle dataset that contains **already-built** tensors. Required files:
   - `note_embeddings.npy`
   - `node_embeddings.npy`
   - `concept_nodes.csv`
   - `encounter_hyperedges.csv`
   - `node_encounter_edges.csv`
   
   These live locally in `mingle-coherent/data/processed/`. **Do not** upload `clinical_notes.csv` (huge, not needed for training).
3. Do **not** rebuild from raw FHIR on Kaggle unless you have to — note embedding is hours even with GPU.

Outputs (`/kaggle/working/checkpoints` and `experiments`) download from the notebook Output tab.

In [ ]:
# Which experiments to run (uncheck if a Kaggle session is short)
RUN_VANILLA = True
RUN_WEIGHTED_50 = True
RUN_FOCAL = True
RUN_NOTE_ABLATION = True

REPO_URL = "https://github.com/ARYANRAJ1121/IITD-Graphical-Neural-Network.git"

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path(".").resolve()
print("kaggle:" , ON_KAGGLE, "work:", WORK)

pkgs = ["PyYAML", "orjson", "sentence-transformers", "gensim", "networkx"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available() or not ON_KAGGLE, "Turn on GPU in Kaggle notebook settings."

In [ ]:
def _git_update(repo: Path) -> None:
    if not ON_KAGGLE or not (repo / ".git").exists():
        return
    try:
        subprocess.check_call(["git", "-C", str(repo), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(repo), "reset", "--hard", "origin/main"])
        print("repo updated from origin/main")
    except subprocess.CalledProcessError as err:
        print("git update skipped:", err)

def find_repo() -> Path:
    marker = Path("src") / "training" / "train_vanilla_bce.py"
    roots = [Path.cwd(), WORK]
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    for root in roots:
        if not root.exists():
            continue
        if (root / marker).exists():
            repo = root.resolve()
            _git_update(repo)
            return repo
        for hit in root.rglob("train_vanilla_bce.py"):
            if hit.parent.name == "training" and hit.parent.parent.name == "src":
                repo = hit.parents[2].resolve()
                _git_update(repo)
                return repo
    dest = WORK / "IITD-Graphical-Neural-Network"
    if dest.exists():
        shutil.rmtree(dest)
    url = "https://github.com/ARYANRAJ1121/IITD-Graphical-Neural-Network.git"
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest.resolve()

REPO = find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo:", REPO)

In [ ]:
def find_processed() -> Path:
    roots = [Path("/kaggle/input"), WORK, REPO / "data" / "processed", Path.cwd() / "data" / "processed"]
    found = []
    for root in roots:
        if not root.exists():
            continue
        for npy in root.rglob("note_embeddings.npy"):
            d = npy.parent
            need = ["node_embeddings.npy", "concept_nodes.csv", "encounter_hyperedges.csv", "node_encounter_edges.csv"]
            if all((d / n).exists() for n in need):
                found.append(d)
    if not found:
        raise FileNotFoundError(
            "Upload data/processed as a Kaggle dataset (npy + csv listed in the first cell)."
        )
    return found[0]

PROCESSED = find_processed()
print("processed:", PROCESSED)
for p in PROCESSED.iterdir():
    if p.suffix in {".npy", ".csv", ".json"} and p.name != "clinical_notes.csv":
        print(f"  {p.name:40s} {p.stat().st_size/1e6:.1f} MB")

In [ ]:
from src.config import load_config
from src.data.processed_bundle import load_processed_bundle
from src.training.train_vanilla_bce import train_vanilla_bce
from src.training.train_weighted_bce import train_weighted_bce
from src.training.train_focal_loss import train_focal_loss
from src.training.train_note_ablation import train_note_ablation

config = load_config(REPO / "configs" / "base.yaml")
config["data"]["processed_dir"] = str(PROCESSED)
out_ckpt = WORK / "checkpoints" if ON_KAGGLE else REPO / "data" / "processed" / "checkpoints"
out_exp = WORK / "experiments" if ON_KAGGLE else REPO / "experiments"
out_ckpt.mkdir(parents=True, exist_ok=True)
out_exp.mkdir(parents=True, exist_ok=True)
config["training"]["checkpoint_dir"] = str(out_ckpt)

bundle = load_processed_bundle(
    PROCESSED,
    int(config["training"]["seed"]),
    tuple(config["training"]["patient_split"]),
)
if bundle is None:
    raise SystemExit("Processed tensors incomplete.")
print("bundle source:", bundle.source, "notes:", tuple(bundle.note_semantics.shape))

## Vanilla BCE (paper loss, 20 epochs)

In [ ]:
results = {}
if RUN_VANILLA:
    results["vanilla"] = train_vanilla_bce(config, bundle, out_exp / "vanilla_bce_training_report.md")
    t = results["vanilla"]["test_metrics"]
    print("vanilla test macro-AUPRC", t["macro_auprc"], "micro-F1", t["micro_f1"])
else:
    print("skipped vanilla")

## Weighted BCE (50 epochs, train-only class weights)

Reloads the bundle so later ablations do not reuse a mutated copy.

In [ ]:
def fresh_bundle():
    b = load_processed_bundle(
        PROCESSED,
        int(config["training"]["seed"]),
        tuple(config["training"]["patient_split"]),
    )
    if b is None:
        raise SystemExit("Processed tensors incomplete.")
    return b

if RUN_WEIGHTED_50:
    results["weighted"] = train_weighted_bce(
        config,
        fresh_bundle(),
        out_exp / "weighted_bce_convergence_report.md",
        epochs=50,
        checkpoint_name="weighted_bce_converged_best.pt",
        history_name="weighted_bce_converged_history.json",
        early_stop_patience=10,
        experiment_name="weighted_bce_convergence",
        compare_weighted_bce_20=True,
    )
    t = results["weighted"]["test_metrics"]
    print("weighted test macro-AUPRC", t["macro_auprc"], "micro-F1", t["micro_f1"])
else:
    print("skipped weighted")

## Focal loss (γ=2, no class weights, 50 epochs)

In [ ]:
if RUN_FOCAL:
    results["focal"] = train_focal_loss(
        config,
        fresh_bundle(),
        out_exp / "focal_loss_report.md",
        epochs=50,
        checkpoint_name="focal_loss_best.pt",
        history_name="focal_loss_history.json",
        early_stop_patience=10,
    )
    t = results["focal"]["test_metrics"]
    print("focal test macro-AUPRC", t["macro_auprc"], "micro-F1", t["micro_f1"])
else:
    print("skipped focal")

## Note ablation (zeros for $N_e$, weighted BCE, 50 epochs)

In [ ]:
if RUN_NOTE_ABLATION:
    results["notes"] = train_note_ablation(
        config,
        fresh_bundle(),
        out_exp / "note_ablation_report.md",
    )
    t = results["notes"]["test_metrics"]
    print("note-ablation test macro-AUPRC", t["macro_auprc"], "micro-F1", t["micro_f1"])
else:
    print("skipped note ablation")

## Comparison

In [ ]:
rows = []
for name, payload in results.items():
    t = payload["test_metrics"]
    rows.append({
        "run": name,
        "best_epoch": payload.get("best_epoch"),
        "bce": round(t["bce"], 6),
        "macro_auprc": t["macro_auprc"],
        "micro_auprc": t["micro_auprc"],
        "macro_auroc": t["macro_auroc"],
        "micro_f1": t["micro_f1"],
        "macro_f1": t["macro_f1"],
    })
import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(out_exp / "kaggle_comparison.csv", index=False)
print("wrote", out_exp / "kaggle_comparison.csv")
print("checkpoints", list(out_ckpt.glob("*.pt")))